# Exploratory data analysis

The target is weekly total sales for each store. The last 26 weeks are reserved for validation and test, so sales plots use only the earlier period.

In [ ]:
from pathlib import Path
import sys

root = Path.cwd()
if not (root / 'src').exists():
    root = root.parent
sys.path.insert(0, str(root))

import pandas as pd
from IPython.display import display, Image
from src.data import DATA_PATH, read_sales, weekly_sales
from src.eda import analyze, OUTPUT

## Raw data and quality checks

In [ ]:
raw = pd.read_csv(DATA_PATH)
display(raw.head())
display(raw.isna().sum())
print('Duplicate keys:', raw.duplicated(['Store', 'Dept', 'Date']).sum())
weekly = weekly_sales(read_sales(DATA_PATH))
weekly.head()

## Distribution, trend and seasonality

Outliers are identified by the IQR rule within each store, not removed automatically.

In [ ]:
summary = analyze(weekly)
display(summary)
for filename in ['sales_vs_time.png', 'sales_distribution.png', 'monthly_seasonality.png', 'holiday_sales.png']:
    display(Image(filename=str(OUTPUT / filename)))

## Feature engineering

Lag and moving-average values are computed from earlier observations only. A day-of-week feature is included for completeness, but it is constant in this weekly dataset.

In [ ]:
from src.features import make_features, split_dates, FEATURE_COLUMNS
features = make_features(weekly)
train, validation, test = split_dates(features)
display(train[FEATURE_COLUMNS].head())
for name, part in [('Train', train), ('Validation', validation), ('Test', test)]:
    print(name, part['Date'].min().date(), part['Date'].max().date(), len(part))

## Interpretation to complete after execution

Describe the strongest seasonal patterns, the holiday effect and whether flagged peaks appear plausible. Do not use test-period results to change feature choices.